# ShellWhisperer v2 — Fine-Tuning Notebook

**Base Model:** `unsloth/qwen2.5-coder-1.5b`

**Key Upgrades from v1:**
- LoRA rank **128** (was 16) — massively more capacity
- LoRA alpha **256** — stronger adaptation signal
- All 7 target modules (q, k, v, o, gate, up, down projections)
- Mix A dataset: **47,824 examples** from fable5-dataset + fableforge

**Runtime:** ~2-3 hours on free Colab T4 GPU

---

## Step 1 — Install Dependencies

Unsloth provides 2x faster training with 70% less memory via Flash Attention and custom CUDA kernels.

In [ ]:
%%capture
# Unsloth + dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

# For GGUF export
!pip install llama_cpp_python

## Step 2 — HuggingFace Login

Store your token in Colab secrets:
1. Click the 🔑 icon in the left sidebar
2. Add a secret named `HF_TOKEN`
3. Paste your HuggingFace write token as the value

This keeps your token out of the notebook source.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to HuggingFace")

## Step 3 — Load Base Model with Unsloth

Loading `unsloth/qwen2.5-coder-1.5b` with 4-bit quantization (NF4) to fit on T4's 16GB VRAM.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/qwen2.5-coder-1.5b"
MAX_SEQ_LENGTH = 4096
DTYPE = None  # Auto-detect; Unsloth handles it
LOAD_IN_4BIT = True  # 4-bit quantization for T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Model dtype: {model.dtype})
print(f"Vocab size: {len(tokenizer))")

## Step 4 — Apply LoRA Adapters

**v2 Upgrade:** Rank 128 (8x increase), Alpha 256, targeting all projection modules.

This gives the adapter significantly more expressive power for shell command generation.

In [ ]:
LORA_RANK = 128
LORA_ALPHA = 256
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA Rank: {LORA_RANK}")
print(f"LoRA Alpha: {LORA_ALPHA}")
print(f"Target modules: {TARGET_MODULES}")
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 5 — Load & Merge Datasets (Mix A)

Loading **fable5-dataset** and **fableforge** from the King3Djbl HuggingFace org.
These combined form Mix A: **47,824 examples**.

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds_fable5 = load_dataset("King3Djbl/fable5-dataset", split="train")
ds_fableforge = load_dataset("King3Djbl/fableforge", split="train")

print(f"fable5-dataset: {len(ds_fable5)} examples")
print(f"fableforge: {len(ds_fableforge)} examples")

dataset = concatenate_datasets([ds_fable5, ds_fableforge])
dataset = dataset.shuffle(seed=3407)

print(f"\nMix A total: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample:")
print(dataset[0])

## Step 6 — Format Dataset for Training

ShellWhisperer uses the Alpaca-style prompt format:
```
### Instruction:
{natural language shell request}

### Response:
{shell command}
```

We auto-detect the column mapping from whatever the datasets provide.

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

# Detect column names
cols = dataset.column_names
instruction_col = None
output_col = None
input_col = None

for col in cols:
    lower = col.lower()
    if lower in ("instruction", "prompt", "question", "input", "query"):
        instruction_col = col
    elif lower in ("output", "response", "answer", "completion", "command", "shell_command"):
        output_col = col
    elif lower in ("input", "context"):
        input_col = col

print(f"Instruction column: {instruction_col}")
print(f"Output column: {output_col}")
print(f"Input column: {input_col}")

def formatting_prompts_func(examples):
    instructions = examples[instruction_col]
    outputs = examples[output_col]
    inputs = examples.get(input_col, [""] * len(instructions)) if input_col else [""] * len(instructions)

    texts = []
    for instruction, output, inp in zip(instructions, outputs, inputs):
        if inp and inp.strip():
            text = ALPACA_PROMPT.format(f"{instruction}\n{inp}", output) + EOS_TOKEN
        else:
            text = ALPACA_PROMPT.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
)

print(f"Formatted dataset: {len(dataset):,} examples")
print(f"\nSample formatted text (first 500 chars):\n{dataset[0]['text'][:500]}")

## Step 7 — Configure Training

Using SFTTrainer from `trl` with Unsloth optimizations.

| Parameter | Value | Why |
|---|---|---|
| Epochs | 3 | Enough for convergence on 47k examples |
| Batch size | 4 | Fits T4 VRAM |
| Grad accum | 4 | Effective batch size = 16 |
| LR | 2e-4 | Standard for LoRA rank 128 |
| Warmup ratio | 0.1 | Stable start |
| Scheduler | Cosine | Smooth decay |

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LR_SCHEDULER = "cosine"
SEED = 3407
SAVE_STEPS = 500
LOG_STEPS = 10

OUTPUT_DIR = "outputs/shellwhisperer-v2"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHEDULER,
        seed=SEED,
        fp16=not FastLanguageModel.get_accelerate_is_bf16(),
        bf16=FastLanguageModel.get_accelerate_is_bf16(),
        logging_steps=LOG_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        optim="adamw_8bit",
        report_to="none",
    ),
)

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION
total_steps = len(dataset) // effective_batch * EPOCHS
print(f"Effective batch size: {effective_batch}")
print(f"Estimated total steps: {total_steps}")
print(f"Estimated training time on T4: ~{total_steps * 2 // 60} minutes")

## Step 8 — Train!

This will take ~2-3 hours on a T4 GPU. Monitor the loss curve.

In [ ]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM: {gpu_stats.total_mem / 1024**3:.1f} GB")
print(f"Starting training...\n")

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")>

## Step 9 — Quick Inference Test

Test the fine-tuned model before pushing.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Find all Python files larger than 1MB in the current directory",
    "Kill the process listening on port 8080",
    "Create a tar archive of all .log files modified in the last 7 days",
    "Show disk usage sorted by size, human readable, top 10",
    "Replace all occurrences of 'TODO' with 'FIXME' recursively in .js files",
]

for prompt in test_prompts:
    inputs = tokenizer(
        [ALPACA_PROMPT.format(prompt, "")],
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("### Response:")[-1].strip()
    print(f"Q: {prompt}")
    print(f"A: {response}\n")

## Step 10 — Push to HuggingFace

Upload the LoRA adapter to `King3Djbl/shellwhisperer-1.5b-v2`.

In [ ]:
HF_REPO = "King3Djbl/shellwhisperer-1.5b-v2"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"LoRA adapter pushed to: https://huggingface.co/{HF_REPO}")

## Step 11 — Export to GGUF for Ollama

Export the merged model as GGUF (Q4_K_M quantization) and push to HuggingFace.

Then create a Modelfile to use with Ollama.

In [ ]:
GGUF_REPO = f"{HF_REPO}-GGUF"

# Save merged model locally first
model.save_pretrained_merged(
    "outputs/shellwhisperer-v2-merged",
    tokenizer,
    save_method="merged_16bit",
)

print("Merged 16-bit model saved locally")

In [ ]:
# Export to GGUF - Q4_K_M for best quality/size balance
model.save_pretrained_gguf(
    "outputs/shellwhisperer-v2-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

# Push GGUF to HuggingFace
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"GGUF pushed to: https://huggingface.co/{GGUF_REPO}")

## Step 12 — Ollama Integration

After the GGUF is on HuggingFace, you can pull it into Ollama:

```bash
# Option 1: Create Modelfile and build locally
ollama create shellwhisperer-v2 -f Modelfile

# Option 2: Pull from HuggingFace directly
ollama pull hf.co/King3Djbl/shellwhisperer-1.5b-v2-GGUF
```

Save the Modelfile below and run the Ollama command locally (not in Colab).

In [ ]:
modelfile_content = '''FROM hf.co/King3Djbl/shellwhisperer-1.5b-v2-GGUF

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_predict 256

SYSTEM """You are ShellWhisperer v2, an expert at translating natural language into precise shell commands. Given a description of what someone wants to do, you output exactly the shell command(s) needed. Prefer one-liners with pipes. Use standard POSIX tools. No explanations, just the command."""

TEMPLATE """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{{ .Prompt }}

### Response:
{{ .Response }}"""
'''

print(modelfile_content)
print("\nSave this as 'Modelfile' and run: ollama create shellwhisperer-v2 -f Modelfile")

## Step 13 — (Optional) Export Full Merged Model

Push the full merged 16-bit model to HuggingFace for downstream use.

In [ ]:
MERGED_REPO = f"{HF_REPO}-merged-16bit"

model.push_to_hub_merged(
    MERGED_REPO,
    tokenizer,
    save_method="merged_16bit",
)

print(f"Full merged model pushed to: https://huggingface.co/{MERGED_REPO}")
print("\n=== ShellWhisperer v2 Training Complete ===")
print(f"LoRA adapter:  https://huggingface.co/{HF_REPO}")
print(f"GGUF:          https://huggingface.co/{GGUF_REPO}")
print(f"Merged 16-bit: https://huggingface.co/{MERGED_REPO}")

---

## Summary

| Artifact | Location |
|---|---|
| LoRA Adapter | `King3Djbl/shellwhisperer-1.5b-v2` |
| GGUF (Q4_K_M) | `King3Djbl/shellwhisperer-1.5b-v2-GGUF` |
| Merged 16-bit | `King3Djbl/shellwhisperer-1.5b-v2-merged-16bit` |
| Ollama | `ollama pull hf.co/King3Djbl/shellwhisperer-1.5b-v2-GGUF` |

### Key v2 Improvements
- **8x LoRA capacity** (rank 128 vs 16)
- **7/7 target modules** (all projections)
- **47,824 training examples** (Mix A)
- **Cosine LR schedule** with warmup
- **Packing enabled** for efficient training